In [0]:
WITH t AS (
  SELECT 
  m.sales_subregion_level_3 as BU3
  , b.account_executive as ae

  /* Uncomment if you need account level data */
  -- , b.last_solution_architect_engaged as sa
  -- , m.account_name
  -- , m.arr_band
  -- , b.t3m_annualized
  /* ***** */
  , m.fiscal_year as fiscal_year
  --, SUM(case when m.submitted_ae_forecast is null then m.dbu_dollars else m.submitted_ae_forecast end) as ae_forecast
  , null as ae_forecast -- the coliumn 'forecast_ae' only includes closed months
  , SUM(m.dbu_dollars) as dbu_dollars
  , SUM(m.dbu_dollar_target) as dbu_dollar_target
  --, SUM(m.submitted_ae_forecast) as submitted_ae_forecast_last_quarter
  , SUM(m.uc_dbu_dollars) as dbu_dollars_uc
  , SUM(m.dbsql_dbu_dollars) AS dbu_dollars_sql
  , SUM(m.genai_all_dbu_dollars) AS dbu_dollars_genai
  , SUM(m.serverless_dbu_dollars) AS dbu_dollars_serverless
  , SUM(m.serverless_jobs_dbu_dollars) AS dbu_dollars_serverless_jobs
  , SUM(m.dbsql_serverless_dbu_dollars) AS dbu_dollars_serverless_sql
  , SUM(m.lakeflow_connect_dbu_dollars) AS lakeflow_connect_dbu_dollars
  , SUM(m.lakeflow_pipeline_dbu_dollars) AS lakeflow_pipeline_dbu_dollars
  , SUM(m.serverless_real_time_inference_dbu_dollars) AS dbu_dollars_serverless_realtimeinference
  , SUM(m.serverless_all_purpose_dbu_dollars) AS dbu_dollars_serverless_all_purpose
  , SUM(m.sku_type_automated_dbu_dollars) AS dbu_dollars_sku_type_automated
  , SUM(m.sku_type_interactive_dbu_dollars) AS dbu_dollars_sku_type_interactive
  , SUM(m.azure_dbu_dollars) AS dbu_dollars_azure
  , SUM(m.aws_dbu_dollars) AS dbu_dollars_aws
  , SUM(m.gcp_dbu_dollars) AS dbu_dollars_gcp
  FROM gtm_gold.account_consumption_monthly as m
  --FROM main.gtm_data.c360_consumption_account_monthly as m -- LEGACY
  LEFT OUTER JOIN main.gtm_silver.account_dim as b
  ON m.account_id = b.account_id
  WHERE m.sales_subregion_level_2 = 'Italy'
  --AND b.account_executive = 'Paolo Garzone'
  AND fiscal_year IN (2025, 2026) -- only completed quarter from curr_fy fiscal and prev_fyious fiscal
  GROUP BY ALL
)


SELECT

/***** YoY ANALYSIS *****/
BU3
, ae
  /* Uncomment if you need account level data */
--, sa
--, account_name
--, arr_band
--, t3m_annualized
  /* ***** */

, SUM(curr_fy_ae_forecast) AS curr_fy_ae_forecast
, SUM(curr_fy_dbu_dollar_target) as curr_fy_dbu_dollar_target
, SUM(try_divide(curr_fy_ae_forecast, curr_fy_dbu_dollar_target)) as curr_fy_forecast_vs_target


, SUM(curr_fy_dbu_dollars) as curr_fy_dbu_dollars
, SUM(prev_fy_dbu_dollars) as prev_fy_dbu_dollars
, SUM(curr_fy_dbu_dollars - prev_fy_dbu_dollars) as incr_dbu_dollars
, SUM(try_divide((curr_fy_dbu_dollars - prev_fy_dbu_dollars), prev_fy_dbu_dollars)) as yoy_growth_actual
, SUM(try_divide((curr_fy_ae_forecast - prev_fy_dbu_dollars), prev_fy_dbu_dollars)) as yoy_growth_forecasted

--dbu_dollars_uc
, SUM(curr_fy_dbu_dollars_uc) as curr_fy_dbu_dollars_uc
, SUM(TRY_DIVIDE(curr_fy_dbu_dollars_uc, curr_fy_dbu_dollars*.8)) as uc_percent_of_dbu_dollars
, SUM(curr_fy_dbu_dollars_uc - prev_fy_dbu_dollars_uc) as incr_dbu_dollars_uc
, SUM(try_divide((curr_fy_dbu_dollars_uc - prev_fy_dbu_dollars_uc), prev_fy_dbu_dollars_uc)) as yoy_growth_uc

--dbu_dollars_sql
, SUM(curr_fy_dbu_dollars_sql) as curr_fy_dbu_dollars_sql
, SUM(TRY_DIVIDE(curr_fy_dbu_dollars_sql, curr_fy_dbu_dollars)) as sql_percent_of_dbu_dollars
, SUM(curr_fy_dbu_dollars_sql - prev_fy_dbu_dollars_sql) as incr_dbu_dollars_sql
, SUM(try_divide((curr_fy_dbu_dollars_sql - prev_fy_dbu_dollars_sql), prev_fy_dbu_dollars_sql)) as yoy_growth_sql

--dbu_dollars_genai
, SUM(curr_fy_dbu_dollars_genai) as curr_fy_dbu_dollars_genai
, SUM(TRY_DIVIDE(curr_fy_dbu_dollars_genai, curr_fy_dbu_dollars)) as genai_percent_of_dbu_dollars
, SUM(curr_fy_dbu_dollars_genai - prev_fy_dbu_dollars_genai) as incr_dbu_dollars_genai
, SUM(try_divide((curr_fy_dbu_dollars_genai - prev_fy_dbu_dollars_genai), prev_fy_dbu_dollars_genai)) as yoy_growth_genai

--dbu_dollars_serverless
, SUM(curr_fy_dbu_dollars_serverless) as curr_fy_dbu_dollars_serverless
, SUM(TRY_DIVIDE(curr_fy_dbu_dollars_serverless, curr_fy_dbu_dollars)) as serverless_percent_of_dbu_dollars
, SUM(curr_fy_dbu_dollars_serverless - prev_fy_dbu_dollars_serverless) as incr_dbu_dollars_serverless
, SUM(try_divide((curr_fy_dbu_dollars_serverless - prev_fy_dbu_dollars_serverless), prev_fy_dbu_dollars_serverless)) as yoy_growth_serverless

--lakeflow_connect_dbu_dollars
, SUM(curr_fy_lakeflow_connect_dbu_dollars) as curr_fy_lakeflow_connect_dbu_dollars
, SUM(TRY_DIVIDE(curr_fy_lakeflow_connect_dbu_dollars, curr_fy_dbu_dollars)) as lakeflow_connect_percent_of_dbu_dollars
, SUM(curr_fy_lakeflow_connect_dbu_dollars - prev_fy_lakeflow_connect_dbu_dollars) as incr_lakeflow_connect_dbu_dollars
, SUM(try_divide((curr_fy_lakeflow_connect_dbu_dollars - prev_fy_lakeflow_connect_dbu_dollars), prev_fy_lakeflow_connect_dbu_dollars)) as yoy_growth_lakeflow_connect

--lakeflow_pipeline_dbu_dollars
, SUM(curr_fy_lakeflow_pipeline_dbu_dollars) as curr_fy_lakeflow_pipeline_dbu_dollars
, SUM(TRY_DIVIDE(curr_fy_lakeflow_pipeline_dbu_dollars, curr_fy_dbu_dollars)) as lakeflow_pipeline_percent_of_dbu_dollars
, SUM(curr_fy_lakeflow_pipeline_dbu_dollars - prev_fy_lakeflow_pipeline_dbu_dollars) as incr_lakeflow_pipeline_dbu_dollars
, SUM(try_divide((curr_fy_lakeflow_pipeline_dbu_dollars - prev_fy_lakeflow_pipeline_dbu_dollars), prev_fy_lakeflow_pipeline_dbu_dollars)) as yoy_growth_lakeflow_pipeline

FROM t  

PIVOT (
  SUM(ae_forecast) AS ae_forecast
  , SUM(dbu_dollar_target) as dbu_dollar_target
  , SUM(dbu_dollars) as dbu_dollars
  , SUM(dbu_dollars_uc) as dbu_dollars_uc
  , SUM(dbu_dollars_sql) AS dbu_dollars_sql
  , SUM(dbu_dollars_genai) as dbu_dollars_genai
  , SUM(dbu_dollars_serverless) as dbu_dollars_serverless
  , SUM(dbu_dollars_serverless_jobs) as dbu_dollars_serverless_jobs
  , SUM(dbu_dollars_serverless_sql) as dbu_dollars_serverless_sql
  , SUM(dbu_dollars_serverless_realtimeinference) as dbu_dollars_serverless_realtimeinference
  , SUM(lakeflow_connect_dbu_dollars) as lakeflow_connect_dbu_dollars
  , SUM(lakeflow_pipeline_dbu_dollars) as lakeflow_pipeline_dbu_dollars
  , SUM(dbu_dollars_serverless_all_purpose) as dbu_dollars_serverless_realtimeinference
  , SUM(dbu_dollars_serverless_all_purpose) as dbu_dollars_serverless_all_purpose
  , SUM(dbu_dollars_sku_type_automated) as dbu_dollars_sku_type_automated
  , SUM(dbu_dollars_sku_type_interactive) as dbu_dollars_sku_type_interactive
  , SUM(dbu_dollars_azure) as dbu_dollars_azure
  , SUM(dbu_dollars_aws) as dbu_dollars_aws
  , SUM(dbu_dollars_gcp) as dbu_dollars_gcp
    FOR fiscal_year IN (2025 as prev_fy, 2026 as curr_fy)
)
GROUP BY ALL


